# Building an Intelligent E-Commerce Support Team with AutoGen and Moss

Welcome to the AutoGen + Moss Cookbook! 

When building production support systems, answering customer inquiries often requires pulling data from several different departments. For example, a question like "My item arrived damaged, can I get a replacement?" inherently requires verifying the return policy, checking replacement inventory, and estimating shipping speeds.

In this cookbook, we will build a **dynamic, multi-agent AI customer support team** designed to resolve complex, multi-step customer issues autonomously. 

We will demonstrate how to:
1. **Implement Intelligent Agent Routing:** Use AutoGen's `SelectorGroupChat` to create an LLM "manager" that dynamically hands off the conversation to specialized agents based on user intent.
2. **Equip Agents with Specialized Tools:** Give our `AssistantAgents` specific Moss search capabilities so they only retrieve data relevant to their role.
3. **Instrument Tool Performance:** Benchmark the latency of semantic search calls to ensure that the system remains snappy even during complex multi-agent reasoning steps.

By the end of this guide, you will have a functional 4-agent team (`product_agent`, `shipping_agent`, `returns_agent`, and `summary_agent`) collaborating seamlessly to resolve real-world e-commerce queries.

In [40]:
!uv pip install -qU "autogen-agentchat" "autogen-ext[openai]" "inferedge-moss" "python-dotenv"

In [41]:
import os
import time
import json
from dotenv import load_dotenv
from inferedge_moss import MossClient, DocumentInfo, QueryOptions
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.model_context import BufferedChatCompletionContext
load_dotenv()

moss_client = MossClient(
    project_id=os.getenv("MOSS_PROJECT_ID"),
    project_key=os.getenv("MOSS_PROJECT_KEY")
)

model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY")
)

## 1. Instrumenting Moss for Benchmarks

To prove the speed of Moss during multi-agent workflows, we will wrap the search in an instrumentation class. Measuring latency is critical for agentic loops, as slow tools compound exponentially when multiple agents collaborate.

> *Note: If you are running this against the hosted Moss cloud API, expect ~1.2s total latency due to network transit and remote text embedding. For true sub-10ms performance, use a self-hosted local instance of Moss.*

In [42]:
class InstrumentedMossSearch:
    def __init__(self, moss_client):
        self.moss = moss_client
        self.stats = {}
        
    def reset(self):
        self.stats = {}
        
    async def query(self, index_name: str, tool_name: str, query: str, top_k: int = 3):
        if tool_name not in self.stats:
            self.stats[tool_name] = {"calls": 0, "total_ms": 0.0}
            
        start_time = time.perf_counter()
        opts = QueryOptions(top_k=top_k)
        results = await self.moss.query(index_name, query, opts)
        end_time = time.perf_counter()
        
        latency_ms = (end_time - start_time) * 1000
        self.stats[tool_name]["calls"] += 1
        self.stats[tool_name]["total_ms"] += latency_ms
        
        call_num = self.stats[tool_name]["calls"]
        print(f"[MOSS] {tool_name:<15} | call #{call_num} | query: '{query}' | latency: {latency_ms:.2f}ms")
        
        if not results.docs:
            return "No relevant information found."
            
        return "\n\n".join([doc.text for doc in results.docs])

    def print_summary(self):
        print("\n========== MOSS BENCHMARK SUMMARY ==========")
        total_all_ms = 0.0
        for tool, stat in self.stats.items():
            calls = stat["calls"]
            tot = stat["total_ms"]
            avg = tot / calls if calls > 0 else 0
            total_all_ms += tot
            print(f"{tool:<15} : {calls} calls | total: {tot:.2f}ms | avg: {avg:.2f}ms")
        print(f"Total Moss latency across all agents: {total_all_ms:.2f}ms")
        print("=============================================" * 2)
        
moss = InstrumentedMossSearch(moss_client)

## 2. Load and Index Datasets
We create 3 independent indices for our 3 data sources.

In [ ]:
async def setup_indices():
    async def load_index(filename, index_name):
        with open(f"data/{filename}", "r") as f:
            data = json.load(f)
        docs = []
        for i, item in enumerate(data):
            text_repr = ", ".join([f"{k}: {v}" for k, v in item.items()])
            doc_id = item.get("id", f"doc_{i}")
            string_metadata = {k: str(v) for k, v in item.items()}
            docs.append(DocumentInfo(id=doc_id, text=text_repr, metadata=string_metadata))
        await moss_client.create_index(index_name, docs)
        print(f"Indexed {len(docs)} documents into '{index_name}'.")

    await load_index("product_catalog.json", "product_catalog_index")
    await load_index("shipping_policies.json", "shipping_policies_index")
    await load_index("return_policies.json", "returns_index")

# Run once
await setup_indices()

## 3. Agent Tools Setup
We wrap the instrumented moss client into dedicated tools for each agent.

In [43]:
async def product_search(query: str) -> str:
    return await moss.query("product_catalog_index", "product_search", query)

async def shipping_search(query: str) -> str:
    return await moss.query("shipping_policies_index", "shipping_search", query)

async def returns_search(query: str) -> str:
    return await moss.query("returns_index", "returns_search", query)

## 4. Setup Agents and SelectorGroupChat

In [44]:
product_agent = AssistantAgent(
    "product_agent",
    model_client=model_client,
    tools=[product_search],
    system_message="You are a product specialist. Use the 'product_search' tool to look up catalog information before answering. If an item is out of stock, say so clearly.",
    reflect_on_tool_use=True
)

shipping_agent = AssistantAgent(
    "shipping_agent",
    model_client=model_client,
    tools=[shipping_search],
    system_message="You are a shipping specialist. Use the 'shipping_search' tool to look up shipping policies. For damaged item replacements, confirm eligibility with returns_agent first before quoting delivery times.",
    reflect_on_tool_use=True
)

returns_agent = AssistantAgent(
    "returns_agent",
    model_client=model_client,
    tools=[returns_search],
    system_message="You are a returns specialist. Use the 'returns_search' tool to look up return policies. For damaged items requiring replacement shipping, state clearly that shipping_agent will handle the delivery details.",
    reflect_on_tool_use=True
)

summary_agent = AssistantAgent(
    "summary_agent",
    model_client=model_client,
    system_message="You are a customer support summarizer. Combine the responses from product_agent, shipping_agent, and returns_agent into a single, clear, friendly response for the customer. End your message with TERMINATE."
)

termination = TextMentionTermination("TERMINATE")
team = SelectorGroupChat(
    participants=[product_agent, shipping_agent, returns_agent, summary_agent],
    model_client=model_client,
    termination_condition=termination,
    allow_repeated_speaker=False,
    model_context=BufferedChatCompletionContext(buffer_size=5),
)

## 5. Test Queries
Let's execute three distinctive flows.

In [45]:
async def run_query(query: str):
    print(f"\n{'='*50}\nQUERY: {query}\n{'='*50}")
    moss.reset()
    await Console(team.run_stream(task=query))
    moss.print_summary()
    await team.reset()

In [46]:
# Query 1: Single-agent (product only)
await run_query("Is the Ergonomic Office Chair currently in stock and what are its specs?")


QUERY: Is the Ergonomic Office Chair currently in stock and what are its specs?
---------- TextMessage (user) ----------
Is the Ergonomic Office Chair currently in stock and what are its specs?
---------- ToolCallRequestEvent (product_agent) ----------
[FunctionCall(id='call_JTQ1PD15wJwKxB9z6P2GoeOw', arguments='{"query":"Ergonomic Office Chair"}', name='product_search')]
[MOSS] product_search  | call #1 | query: 'Ergonomic Office Chair' | latency: 1189.30ms
---------- ToolCallExecutionEvent (product_agent) ----------
[FunctionExecutionResult(content='id: SKU-002, name: Ergonomic Office Chair, price: 349.99, stock: 0, specs: Lumbar support, adjustable armrests, mesh back, category: Furniture\n\nid: SKU-005, name: Yoga Mat, price: 49.99, stock: 60, specs: 6mm thick, non-slip, eco-friendly TPE, category: Fitness\n\nid: SKU-004, name: Mechanical Keyboard, price: 129.99, stock: 15, specs: TKL layout, Cherry MX Red switches, RGB backlight, category: Electronics', name='product_search', cal

In [47]:
# Query 2: Single-agent (shipping only)
await run_query("What are my shipping options and how much does express delivery cost?")


QUERY: What are my shipping options and how much does express delivery cost?
---------- TextMessage (user) ----------
What are my shipping options and how much does express delivery cost?
---------- ToolCallRequestEvent (shipping_agent) ----------
[FunctionCall(id='call_T8wKTxx7S14vMKYYqsk3NnDR', arguments='{"query":"shipping options and express delivery cost"}', name='shipping_search')]
[MOSS] shipping_search | call #1 | query: 'shipping options and express delivery cost' | latency: 1336.05ms
---------- ToolCallExecutionEvent (shipping_agent) ----------
[FunctionExecutionResult(content='type: Express Shipping, cost: 14.99, eta_days: 1-2, eligible_for: In-stock items only\n\ntype: Standard Shipping, cost: 4.99, eta_days: 5-7, eligible_for: All orders\n\ntype: Free Shipping, cost: 0, eta_days: 5-7, eligible_for: Orders over $75', name='shipping_search', call_id='call_T8wKTxx7S14vMKYYqsk3NnDR', is_error=False)]
---------- TextMessage (shipping_agent) ----------
Here are your shipping op

In [48]:
# Query 3: Multi-agent Coordinated (returns -> shipping -> summary)
await run_query("My headphones arrived damaged. Can I get a replacement and how quickly can it be shipped?")


QUERY: My headphones arrived damaged. Can I get a replacement and how quickly can it be shipped?
---------- TextMessage (user) ----------
My headphones arrived damaged. Can I get a replacement and how quickly can it be shipped?
---------- ToolCallRequestEvent (returns_agent) ----------
[FunctionCall(id='call_IytNqjyCHD2sVgDxzNE56O4S', arguments='{"query":"damaged headphones replacement policy"}', name='returns_search')]
[MOSS] returns_search  | call #1 | query: 'damaged headphones replacement policy' | latency: 1361.69ms
---------- ToolCallExecutionEvent (returns_agent) ----------
[FunctionExecutionResult(content='policy: Damaged Item Return, window_days: 90, condition: Item damaged on arrival or within 90 days of purchase, refund_method: Full refund or free replacement, processing_days: 2\n\npolicy: Electronics Return, window_days: 15, condition: Unopened or confirmed defective, refund_method: Store credit or original payment method, processing_days: 7\n\npolicy: Furniture Return, wi